# 4회차 · Python·FastAPI 기반 AI Agent 서비스 개발

**확정 변경:** Codex Desktop 실습 제외. Python, FastAPI, Pydantic을 활용하여 3회차 Agent 코드를 REST API로 구성합니다.  
**산출물:** `04_agent_web_mvp/app/{tools,llm,agent,main}.py`, API 테스트 결과, 이후 5회차 확장용 프로젝트.



## STEP 1. 프로젝트 저장 위치 선택

In [1]:
from pathlib import Path
import sys
BASE_DIR=Path("/workspace") if Path("/workspace").exists() else Path.cwd()
PROJECT_DIR=BASE_DIR/"04_agent_web_mvp"
(PROJECT_DIR/"app").mkdir(parents=True,exist_ok=True)
print("프로젝트:",PROJECT_DIR)
print("Python:",sys.executable)

프로젝트: /workspace/04_agent_web_mvp
Python: /usr/local/bin/python


## STEP 2. FastAPI 설치 (현재 Jupyter 커널)

In [2]:
%pip install -U "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "pydantic>=2,<3" "httpx>=0.27,<1"

Note: you may need to restart the kernel to use updated packages.


## STEP 3. Custom Tool 코드 생성 — `app/tools.py`
3회차에서 사용한 네 가지 Tool과 Registry를 재사용합니다.

In [3]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/tools.py'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)
    print("수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.")
else:
    file_path.write_text(r'''
"""수업용 Custom Tool: 허용된 연산/명령만 실행한다."""
import ast
import operator as op
import math
from datetime import datetime
from zoneinfo import ZoneInfo

MAX_INPUT = 100
MAX_ABS_RESULT = 1_000_000_000_000
OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
       ast.Div: op.truediv, ast.FloorDiv: op.floordiv, ast.Mod: op.mod,
       ast.USub: op.neg, ast.UAdd: op.pos}

def _calculate(node, depth=0):
    if depth > 15:
        raise ValueError("수식의 깊이가 제한을 초과했습니다.")
    if isinstance(node, ast.Expression):
        return _calculate(node.body, depth+1)
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        value = node.value
    elif isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        value = OPS[type(node.op)](_calculate(node.operand, depth+1))
    elif isinstance(node, ast.BinOp) and type(node.op) in OPS:
        a = _calculate(node.left, depth+1)
        b = _calculate(node.right, depth+1)
        value = OPS[type(node.op)](a, b)
    elif isinstance(node, ast.BinOp) and isinstance(node.op, ast.Pow):
        a = _calculate(node.left, depth+1)
        b = _calculate(node.right, depth+1)
        if abs(a) > 1_000_000 or abs(b) > 8:
            raise ValueError("거듭제곱 범위를 초과했습니다.")
        value = a ** b
    else:
        raise ValueError("허용되지 않은 수식입니다.")
    if not isinstance(value, (int,float,complex)) or isinstance(value, complex):
        raise ValueError("실수 결과만 허용합니다.")
    if not math.isfinite(value) or abs(value) > MAX_ABS_RESULT:
        raise ValueError("결과 범위를 초과했습니다.")
    return value

def calculator(expression: str) -> dict:
    """사칙연산·거듭제곱 계산. Python eval() 사용 금지."""
    try:
        if not isinstance(expression, str) or len(expression) > MAX_INPUT:
            raise ValueError("수식은 길이 100 이하의 문자열이어야 합니다.")
        tree = ast.parse(expression, mode="eval")
        return {"ok": True, "result": _calculate(tree)}
    except (ValueError, SyntaxError, ArithmeticError, TypeError) as e:
        return {"ok": False, "error": str(e)}

def text_length(text: str) -> dict:
    if not isinstance(text, str):
        return {"ok": False, "error": "text는 문자열이어야 합니다."}
    return {"ok": True, "result": len(text)}

def current_time() -> dict:
    return {"ok": True, "result": datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y-%m-%d %H:%M:%S KST")}

def data_summary(values: list) -> dict:
    """수치 목록 개수/합계/평균/최솟값/최댓값 요약."""
    if not isinstance(values, list) or not (1 <= len(values) <= 100):
        return {"ok": False, "error": "values는 1~100개의 숫자 목록이어야 합니다."}
    if any(type(v) not in (int,float) or not math.isfinite(v) or abs(v)>1e9 for v in values):
        return {"ok": False, "error": "유효한 숫자만 사용할 수 있습니다."}
    return {"ok": True, "result": {"count":len(values), "sum":round(sum(values),4),
            "average":round(sum(values)/len(values),4), "min":min(values), "max":max(values)}}

TOOL_REGISTRY = {"calculator": calculator, "text_length": text_length,
                 "current_time": current_time, "data_summary": data_summary}
TOOL_DESCRIPTIONS = {
    "calculator": {"description":"수식을 정확히 계산", "arguments":{"expression":"계산할 수식 문자열"}},
    "text_length": {"description":"문자열 글자 수 계산", "arguments":{"text":"문자열"}},
    "current_time": {"description":"현재 한국 날짜·시각", "arguments":{}},
    "data_summary": {"description":"숫자 목록의 합계·평균·최솟값·최댓값", "arguments":{"values":"숫자 목록, 예: [10,20,30]"}},
}

def execute_tool(tool_name: str, arguments: dict) -> dict:
    if tool_name not in TOOL_REGISTRY:
        return {"ok": False, "error": f"등록되지 않은 Tool: {tool_name}"}
    if not isinstance(arguments, dict):
        return {"ok": False, "error": "arguments는 JSON 객체(dict)여야 합니다."}
    try:
        return TOOL_REGISTRY[tool_name](**arguments)
    except (TypeError, ValueError) as e:
        return {"ok": False, "error": f"인자 오류: {e}"}

    ''', encoding="utf-8")
print("확인:", file_path)


확인: /workspace/04_agent_web_mvp/app/tools.py


## STEP 4. LLM 코드 생성 — `app/llm.py`
모델은 요청이 올 때만 로드하므로 `/health`는 CPU에서도 응답할 수 있습니다.

In [4]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/llm.py'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)
    print("수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.")
else:
    file_path.write_text(r'''
"""RunPod GPU의 Qwen3-8B 로컬 추론. 첫 /agent 요청 시 모델 지연 로딩."""
import os
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-8B"
if Path("/workspace").is_dir():
    os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
_tokenizer = None
_model = None

def gpu_status() -> dict:
    ok = torch.cuda.is_available()
    return {"cuda_available": bool(ok),
            "gpu": torch.cuda.get_device_name(0) if ok else None}

def load_model():
    global _tokenizer, _model
    if _model is None:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA GPU가 없습니다. RunPod GPU에 접속하세요.")
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        _model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, dtype=torch.bfloat16, device_map="auto"
        ).eval()
    return _tokenizer, _model

def generate_llm(messages: list, max_new_tokens: int=256) -> str:
    tokenizer, model = load_model()
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
    n_in = inputs["input_ids"].shape[-1]
    return tokenizer.decode(outputs[0][n_in:], skip_special_tokens=True).strip()

    ''', encoding="utf-8")
print("확인:", file_path)


확인: /workspace/04_agent_web_mvp/app/llm.py


## STEP 5. Agent 제어 코드 생성 — `app/agent.py`
JSON 판단, Tool 실행, 최종 응답을 하나의 함수로 연결합니다.

In [5]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/agent.py'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)
    print("수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.")
else:
    file_path.write_text(r'''
"""LLM의 JSON 결정 → 허용된 Tool 실행 → 최종 응답."""
import json
import re
import time
from .llm import generate_llm
from .tools import TOOL_DESCRIPTIONS, execute_tool

TOOL_SYSTEM_PROMPT = (
    "당신은 도구 선택 AI Agent다. 가능한 행동은 tool 또는 answer뿐이다.\n"
    "필요한 경우 calculator, text_length, current_time, data_summary 중 하나를 선택한다.\n"
    "숫자 목록 통계에는 data_summary를, 계산에는 calculator를 사용한다.\n"
    "도구가 필요 없는 개념 설명은 answer로 반환한다.\n"
    "반드시 Markdown 없는 JSON 객체 하나만 반환한다.\n"
    '도구 호출: {"action":"tool","tool_name":"calculator","arguments":{"expression":"2+2"}}\n'
    '일반 답변: {"action":"answer","answer":"간단한 설명"}\n'
    + "사용 가능한 Tool: " + json.dumps(TOOL_DESCRIPTIONS, ensure_ascii=False)
)

def parse_agent_json(raw: str) -> dict:
    text = raw.strip()
    # Markdown code fence가 있으면 제거해 파싱을 시도하지만, 실패는 그대로 기록한다.
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
        text = re.sub(r"\s*```$", "", text)
    try:
        obj = json.loads(text)
    except json.JSONDecodeError as e:
        return {"action":"error", "error":f"JSON 파싱 실패: {e}"}
    if not isinstance(obj, dict) or obj.get("action") not in ("tool","answer"):
        return {"action":"error", "error":"action은 tool 또는 answer여야 합니다."}
    if obj["action"] == "tool" and (not isinstance(obj.get("tool_name"),str) or not isinstance(obj.get("arguments"),dict)):
        return {"action":"error", "error":"tool_name/arguments 형식 오류"}
    if obj["action"] == "answer" and not isinstance(obj.get("answer"),str):
        return {"action":"error", "error":"answer는 문자열이어야 합니다."}
    return obj

def make_final_answer(query: str, tool_name: str, tool_result: dict) -> str:
    return generate_llm([
        {"role":"system", "content":"아래 Tool 결과에 근거해 짧고 정확하게 한국어로 답하라."},
        {"role":"user", "content":f"질문: {query}\nTool: {tool_name}\n결과: {json.dumps(tool_result,ensure_ascii=False)}"}
    ], max_new_tokens=160)

def run_agent(user_query: str) -> dict:
    start = time.perf_counter()
    raw = generate_llm([
        {"role":"system", "content":TOOL_SYSTEM_PROMPT},
        {"role":"user", "content":user_query}
    ])
    decision = parse_agent_json(raw)
    result = {"status":"error", "used_tool":False, "tool_name":None,
              "arguments":{}, "tool_result":None, "decision":decision,
              "final_answer":"Agent 결정을 처리하지 못했습니다."}
    if decision["action"] == "answer":
        result.update(status="success", final_answer=decision["answer"])
    elif decision["action"] == "tool":
        tool_name = decision["tool_name"]
        arguments = decision["arguments"]
        tool_result = execute_tool(tool_name, arguments)
        result.update(used_tool=True, tool_name=tool_name,
                      arguments=arguments, tool_result=tool_result)
        if tool_result.get("ok"):
            result.update(status="success",
                          final_answer=make_final_answer(user_query, tool_name, tool_result))
        else:
            result.update(status="tool_error",
                          final_answer=f"Tool 실행 오류: {tool_result.get('error')}")
    result["elapsed_ms"] = round((time.perf_counter()-start)*1000, 1)
    return result

    ''', encoding="utf-8")
print("확인:", file_path)


확인: /workspace/04_agent_web_mvp/app/agent.py


## STEP 6. FastAPI 라우트 생성 — `app/main.py`
`GET /`, `GET /health`, `GET /tools`, `POST /agent`를 구현합니다.

In [6]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/main.py'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)
    print("수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.")
else:
    file_path.write_text(r'''
"""4회차: FastAPI Backend. 5회차에 Web UI를 추가한다."""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator
from .agent import run_agent
from .llm import MODEL_ID, gpu_status
from .tools import TOOL_DESCRIPTIONS

app = FastAPI(title="YU AI Agent Backend", version="1.0.0")

class AgentRequest(BaseModel):
    query: str = Field(min_length=1, max_length=1000)

    @field_validator("query")
    @classmethod
    def check_query(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("빈 질문은 허용되지 않습니다.")
        return value.strip()

@app.get("/")
def root():
    return {"message": "AI Agent Backend 정상", "api_docs": "/docs"}

@app.get("/health")
def health():
    return {"status":"ok", "model":MODEL_ID, **gpu_status()}

@app.get("/tools")
def tools():
    return TOOL_DESCRIPTIONS

@app.post("/agent")
def agent(request: AgentRequest):
    try:
        return run_agent(request.query)
    except (RuntimeError, OSError, ValueError) as e:
        raise HTTPException(status_code=503, detail=f"GPU/모델 준비 실패: {e}") from e

    ''', encoding="utf-8")
print("확인:", file_path)


확인: /workspace/04_agent_web_mvp/app/main.py


## STEP 7. requirements.txt, README, 패키지 초기화

In [7]:
if not (PROJECT_DIR/"app"/"__init__.py").exists():
    (PROJECT_DIR/"app"/"__init__.py").write_text("",encoding="utf-8")
else:
    print("기존 파일 보존:", PROJECT_DIR/"app"/"__init__.py")
if not (PROJECT_DIR/"requirements.txt").exists():
    (PROJECT_DIR/"requirements.txt").write_text('fastapi>=0.115,<1\nuvicorn>=0.30,<1\npydantic>=2,<3\nhttpx>=0.27,<1\ntorch\ntransformers>=4.53,<5\naccelerate>=1.0\nsentencepiece\n',encoding="utf-8")
else:
    print("기존 파일 보존:", PROJECT_DIR/"requirements.txt")
if not (PROJECT_DIR/"README.md").exists():
    (PROJECT_DIR/"README.md").write_text('# 4회차: Python·FastAPI 기반 AI Agent Backend\n\n구조: FastAPI → Agent → Qwen3-8B → Custom Tool. **Codex/API 유료키는 사용하지 않습니다.**\n\n1. VS Code Remote-SSH로 RunPod GPU Pod에 연결하고 `/workspace`에서 압축을 풉니다.\n2. `cd /workspace/04_agent_web_mvp`\n3. `python -m pip install -r requirements.txt` (RunPod PyTorch 설치 상태 확인)\n4. `uvicorn app.main:app --host 127.0.0.1 --port 8000`\n5. VS Code의 Ports에서 8000을 포워딩하고 **로컬 PC의 `http://localhost:8000/docs`**로 접속합니다.\n6. GET /health, GET /tools, POST /agent 순서로 테스트합니다. 최초 /agent 호출은 모델을 내려받아 느릴 수 있습니다.\n\n**주의:** RunPod Pod/VS Code의 포트 포워딩 방식은 환경에 따라 다릅니다. 교육용 서버는 인증 없이 인터넷에 직접 공개하지 않습니다.\nHF 모델 캐시는 `/workspace/hf-cache`에 저장됩니다. 영구 저장 여부는 별도로 확인하세요.\n\n오류/검증: CUDA가 없는 환경은 GET /health는 가능하지만 POST /agent는 503. 빈 질문은 422.\n',encoding="utf-8")
else:
    print("기존 파일 보존:", PROJECT_DIR/"README.md")
if not (PROJECT_DIR/".gitignore").exists():
    (PROJECT_DIR/".gitignore").write_text('# 절대 GitHub에 커밋하지 않을 자료\n.env\n.env.*\n*.pem\n*.key\nid_ed25519*\n__pycache__/\n.ipynb_checkpoints/\n*.pyc\nhf-cache/\nmodels/\nruns/\n',encoding="utf-8")
else:
    print("기존 파일 보존:", PROJECT_DIR/".gitignore")
print("프로젝트 구성 파일 준비 완료")

프로젝트 구성 파일 준비 완료


## STEP 8. 소스 문법 검사 및 구조 확인 — GPU 불필요

In [8]:
import ast
for file in sorted((PROJECT_DIR/"app").glob("*.py")):
    ast.parse(file.read_text(encoding="utf-8"))
    print("문법 OK:",file.name)

문법 OK: __init__.py
문법 OK: agent.py
문법 OK: llm.py
문법 OK: main.py
문법 OK: tools.py


## STEP 9. FastAPI TestClient로 API 기본 검사 — GPU 불필요
실제 모델을 부르지 않는 `/health`, `/tools`, 유효성 오류 `422`를 확인합니다.

In [9]:
import sys
from fastapi.testclient import TestClient
# 같은 커널에서 4회차/5회차를 순서대로 열었더라도 올바른 앱을 import합니다.
for module_name in list(sys.modules):
    if module_name == "app" or module_name.startswith("app."):
        sys.modules.pop(module_name, None)
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0,str(PROJECT_DIR))
from app.main import app

with TestClient(app) as client:
    for path in ("/","/health","/tools","/docs"):
        r=client.get(path)
        print(path,"HTTP",r.status_code)
        assert r.status_code==200
    empty=client.post("/agent",json={"query":"   "})
    print("빈 요청:",empty.status_code)
    assert empty.status_code==422
print("API 기본 테스트 통과")

/usr/local/lib/python3.12/dist-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


/ HTTP 200
/health HTTP 200
/tools HTTP 200
/docs HTTP 200
빈 요청: 422
API 기본 테스트 통과


## STEP 10. RunPod에서 FastAPI 실행 (VS Code 터미널)
###(VS Code 터미널과 RUNPOD JupyterLab 터미널)
GPU 확인 후 VS Code **원격 터미널**에서 실행하세요. 실행을 유지해야 브라우저에서 접속할 수 있습니다.

```bash
cd /workspace/04_agent_web_mvp
python -m pip install -r requirements.txt
nvidia-smi
# python -m uvicorn app.main:app --host 127.0.0.1 --port 8000 # VS Code SSH 포트 전달(vscode에서 실행할 때 필요)
python -m uvicorn app.main:app --host 0.0.0.0 --port 8000 #RunPod HTTP 프록시로 브라우저 접속
```

VS Code **PORTS(포트)** 탭에서 `8000`을 포워딩하고, 자신의 Windows 브라우저에서 `http://localhost:8000/docs`로 접속합니다. 공개된 HTTP 포트 개방보다 SSH 포워딩을 권장합니다.

## STEP 11. Swagger UI에서 실제 GPU `/agent` 호출

`POST /agent` → Try it out → 아래 JSON을 입력합니다.

```json
{"query":"33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?"}
```

확인할 값: `status`, `used_tool`, `tool_name`, `arguments`, `tool_result`, `final_answer`, `elapsed_ms`.

- GPU 없이 `/agent` 호출 시 HTTP 503은 예상된 동작입니다.
- 모델 최초 다운로드/로딩 시 응답 시간이 길어질 수 있습니다.
- 웹 서비스를 인증 없이 외부에 공개하지 않습니다.

## STEP 12. CPU만으로 Custom Tool 단위 테스트

In [10]:
from app.tools import execute_tool
assert execute_tool("calculator",{"expression":"33*30"})["result"]==990
assert execute_tool("data_summary",{"values":[10,20,30]})["result"]["average"]==20
assert execute_tool("weather",{})["ok"] is False
print("Tool 단위 테스트 통과")

Tool 단위 테스트 통과


## STEP 13. 다음 5회차 연결
- FastAPI 서버는 `.py` 소스 파일로 실행하고, Notebook은 코드를 생성·검증하는 실습 기록입니다.
- **5회차**에는 `app/static/index.html`, `style.css`, `app.js`를 추가하고 `app/main.py`에 UI 경로를 연결합니다.
- 4회차 결과물: `04_agent_web_mvp/` Backend 프로젝트 및 API 테스트 화면 캡처.